- 실습 기본 환경 설정


In [ ]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

# 12-2 LLM을 미세 조정해 작업에 특화시키기

본 노트북은 본문 12-2절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- 허깅페이스 허브에 등록된 데이터셋 불러오기와 `map()`으로 일괄 전처리
- 사전 학습된 KoBART 모델에 질문을 던져 미세 조정 전 결과 확인
- `Seq2SeqTrainer` 학습 엔진으로 미세 조정
- 미세 조정 전후의 출력 비교([표 12-6])

본문의 모델 15의 구현에 해당되며, 모델 15의 제시문은 다음과 같다.

> **모델 15. 뉴스 요약기 모델**
>
> 사전 학습된 한국어 LLM을 출발점 삼아 뉴스를 요약하는 모델을 만든다. 네이버 뉴스 요약 데이터셋으로 미세 조정한다.

## 데이터셋 준비

- `datasets` 라이브러리의 `load_dataset()`으로 데이터셋을 불러오면 `train`, `validation`, `test` 분할이 함께 들어온다.

In [ ]:
######################################################################################
# 코드 12-8 - 허깅페이스 허브 등록 데이터셋 불러오기
######################################################################################

from datasets import load_dataset

dataset = load_dataset('daekeun-ml/naver-news-summarization-ko')
print(dataset)

TRAIN_SIZE, VALID_SIZE = 500, 100
# 허깅페이스 Dataset의 select() 메서드 - 인덱스 기반 샘플링
train_dataset = dataset['train'].select(range(TRAIN_SIZE))
valid_dataset = dataset['validation'].select(range(VALID_SIZE))

print(f'\n훈련 샘플 수: {len(train_dataset):,}')
print(f'검증 샘플 수: {len(valid_dataset):,}')
print(f'첫 샘플 컬럼: {list(train_dataset[0].keys())}')

- 허깅페이스 `Dataset`은 `map()` 메서드로 콜백 함수를 전체에 적용해 일괄 변환한다.
    - 입력 기사와 정답 요약문을 각각 토큰화하고 길이를 제한한다.

In [ ]:
######################################################################################
# 코드 12-9 - 전처리 콜백 함수 정의와 map() 메서드 적용
######################################################################################

from transformers import AutoTokenizer

MODEL_NAME = 'gogamza/kobart-base-v2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

# 전처리 콜백 함수: examples는 샘플이 아닌 샘플 묶음(map(batched=True))
def preprocess(examples):
    # 뉴스 원문 인코딩: input_ids, attention_mask 컬럼이 생긴다.
    model_inputs = tokenizer(
        examples['document'], max_length=MAX_INPUT_LENGTH,
        truncation=True, padding=False,
    )
    # 요약문 인코딩: input_ids만 labels 컬럼으로 사용한다.
    labels = tokenizer(
        examples['summary'], max_length=MAX_TARGET_LENGTH,
        truncation=True, padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs


# map() 메서드 + 전처리 콜백 함수: 데이터셋 전체에 전처리 적용
tokenized_train = train_dataset.map(            # 훈련 데이터셋
    preprocess, batched=True, remove_columns=train_dataset.column_names,
)
tokenized_valid = valid_dataset.map(            # 검증 데이터셋
    preprocess, batched=True, remove_columns=valid_dataset.column_names,
)

print(f'전처리된 컬럼: {tokenized_train.column_names}')
print(f'첫 샘플 input_ids 길이: {len(tokenized_train[0]["input_ids"])}')
print(f'첫 샘플 labels 길이   : {len(tokenized_train[0]["labels"])}')

## 미세 조정 전 모델의 결과

- 인코더와 디코더를 모두 갖춘 KoBART는 `AutoModelForSeq2SeqLM`으로 불러온다.
- 미세 조정 전후 결과를 같은 방식으로 비교하기 위해 질문 함수를 정의한다.
    - 생성 방식은 빔 너비 4의 빔 서치를 사용하고, 3-gram 반복 금지와 반복 토큰 페널티를 적용한다.

In [ ]:
######################################################################################
# 코드 12-10 - 사전 학습된 KoBART 모델에 질문 던지기(출력 결과는 [표 12-6]에서 확인)
######################################################################################

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print(f'모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}')
print(f'토크나이저 어휘 크기: {tokenizer.vocab_size:,}')

def query_lm(text, model, tokenizer,
             max_input=MAX_INPUT_LENGTH, max_output=MAX_TARGET_LENGTH):
    inputs = tokenizer(
        text, return_tensors='pt', max_length=max_input, truncation=True,
    ).to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_output,
            do_sample=False,            # 탐욕 디코딩(기본값)
            num_beams=4,                # 빔 너비 4의 빔 서치
            early_stopping=True,        # 완성 빔이 num_beams개가 되면 탐색 종료
            no_repeat_ngram_size=3,     # 3-gram 반복 금지
            repetition_penalty=1.2,     # 반복 토큰 생성 페널티
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


# 미세 조정하지 않은 모델의 결과 (이후 비교를 위해 별도 객체로 보관)
model_before = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model_before.eval()
print(query_lm(dataset['test'][0]['document'], model_before, tokenizer))

- 미세 조정하지 않은 모델은 입력 기사의 일부를 그대로 따라 적은 뒤 같은 단어와 구절이 끝없이 반복되는 어색한 출력을 낸다.
    - 요약이라는 의도가 학습에 반영되지 않아서 생긴 결과다.

## 학습 엔진으로 미세 조정

- `transformers` 라이브러리로 불러온 모델을 미세 조정할 때는 학습 루프를 직접 구현하지 않고 학습 엔진 클래스를 사용한다.
    - 본문 [표 12-5]는 모델의 성격에 따라 골라 쓰는 학습 엔진 클래스를 정리한다.
- `Seq2SeqTrainer`는 배치 병합 객체(`DataCollatorForSeq2Seq`)와 학습 설정 객체(`Seq2SeqTrainingArguments`)를 함께 받는다.

In [ ]:
######################################################################################
# 코드 12-11 - 배치 병합 객체와 학습 설정 객체
######################################################################################

from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments

# 배치 단계에서 패딩을 추가하는 배치 병합 객체
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir='../../checkpoint/kobart-news', num_train_epochs=2,
    per_device_train_batch_size=4, per_device_eval_batch_size=4,
    warmup_steps=50,                 # 학습률을 점진적으로 올리는 미니배치 수
    weight_decay=0.01,               # 과적합 방지를 위한 가중치 감쇠 계수
    logging_steps=50,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True,
    predict_with_generate=True,      # 검증 단계에서 generate()로 출력 시퀀스 생성
    fp16=False,                      # CPU/MPS 환경 호환을 위해 비활성.
    # GPU(CUDA) 환경에서는 True -> 학습 속도와 메모리 모두 유리
    report_to='none',                # 외부 로깅 서비스 미사용
)

- 두 객체와 데이터셋을 묶어 `Seq2SeqTrainer`를 만든 뒤 `train()`을 호출하면 학습이 진행된다.

In [ ]:
######################################################################################
# 코드 12-12 - Seq2SeqTrainer 학습 엔진 생성 및 실행
######################################################################################

from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model, args=training_args, train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    processing_class=tokenizer,      # 최근 버전에서 tokenizer -> processing_class
    data_collator=data_collator,
)

# Trainer가 학습 로그(네이티브)를 직접 출력한다. 직접 에포크 print는 만들지 않고,
# 반환된 metrics의 train_runtime으로 전체 학습 시간만 사람이 읽기 좋게 출력한다(§7.7).
result = trainer.train()             # 미세 조정 학습 실행
print(f'전체 학습 시간: {result.metrics["train_runtime"]:.1f}초')

- `Trainer`는 학습 로그를 `trainer.state.log_history`에 스텝 단위로 쌓는다. 이를 꺼내 학습 곡선을 그린다.

In [ ]:
# 참고 - log_history에서 손실을 뽑아 학습 곡선을 그린다 (§7.7)
train_curve = [(rec['step'], rec['loss'])
               for rec in trainer.state.log_history if 'loss' in rec]
eval_curve = [(rec['step'], rec['eval_loss'])
              for rec in trainer.state.log_history if 'eval_loss' in rec]

viz.plot_histories(
    {'훈련 손실': train_curve, '검증 손실': eval_curve},
    title='KoBART 미세 조정 학습 곡선', x_label='스텝', y_label='손실',
)

## 미세 조정 전후의 비교([표 12-6])

- 학습 전후의 모델로 같은 뉴스 기사를 요약해 비교한다.

In [ ]:
# 참고 - 미세 조정 전후의 요약 결과 비교 ([표 12-6])
model_after = trainer.model
model_after.eval()

for i in range(3):
    sample = dataset['test'][i]
    before = query_lm(sample['document'], model_before, tokenizer)
    after = query_lm(sample['document'], model_after, tokenizer)
    print(f'\n[뉴스 원문 {i + 1}] (입력 {len(sample["document"])}자)')
    print(f'  학습 전 출력(50자): {before[:50]}... ({len(before)}자)')
    print(f'  학습 후 출력(50자): {after[:50]}... ({len(after)}자)')

- 마지막은 뉴스가 아닌 일반 텍스트를 입력한 결과다.
    - 학습 전 모델은 입력을 그대로 따라 적지만, 미세 조정한 모델은 분야가 달라도 요약하려고 시도한다.

In [ ]:
# 참고 - 분야 밖 일반 텍스트 입력 결과 ([표 12-6] 마지막 행)
sample_text = (
    '이 자라야 한다는 것은 내가 아니라 장차 내 아내가 될 점순이의 키 말이다. '
    '내가 여기에 와서 돈 한푼 안 받고 일하기를 삼 년 하고 꼬박이 일곱 달 동안을 했다. '
    '그런데도 미처 못 자랐다니까 이 키는 언제야 자라는 겐지 짜장 영문 모른다. '
    '일을 좀더 잘해야 한다든지 혹은 밥을 (많이 먹는다고 노상 걱정이니까) 좀 덜 먹어야 '
    '한다든지 하면 나도 얼마든지 할 말이 많다. 하지만 점순이가 아직 어리니까 더 자라야 '
    '한다는 여기에는 어째 볼 수 없이 그만 벙벙하고 만다. 이래서 나는 애최 계약이 잘못된 '
    '걸 알았다. 이태면 이태, 삼 년이면 삼 년, 기한을 딱 작정하고 일을 했어야 원 할 것이다. '
    '덮어놓고 딸이 자라는 대로 성례를 시켜 주마, 했으니 누가 늘 지키고 섰는 것도 아니고 '
    '그 키가 언제 자라는지 알 수 있는가. 그리고 난 사람의 키가 무럭무럭 자라는 줄만 알았지 '
    '붙박이 키에 모로만 벌어지는 몸도 있는 것을 누가 알았으랴. 때가 되면 장인님이 어련하랴 '
    '싶어서 군소리 없이 꾸벅꾸벅 일만 해왔다.'    
)
before = query_lm(sample_text, model_before, tokenizer)
after = query_lm(sample_text, model_after, tokenizer)
print(f'입력 ({len(sample_text)}자): {sample_text[:50]}...')
print(f'학습 전 출력 ({len(before)}자): {before[:50]}...')
print(f'학습 후 출력 ({len(after)}자): {after[:50]}...')

## 정리

- 허깅페이스 `datasets`와 `Dataset.map()`으로 데이터 준비 과정을 간결하게 처리할 수 있다.
- `Trainer` 계열 학습 엔진을 사용하면 학습 루프를 직접 구현하지 않아도 된다. 모델의 성격에 맞는 엔진 클래스를 골라야 한다.
- 미세 조정은 모델에 '의도'를 심는 과정이다. 같은 모델이라도 미세 조정 전후의 출력 성격이 크게 달라진다.